In [1]:
import numpy as np
import pandas as pd

import re #आपल्याला special characters आणि numbers काढायचे आहेत.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
resumes=pd.read_csv('Resume.csv')

In [3]:
jobs=pd.read_csv('job_sample.csv')

In [4]:
resumes.info()

<class 'pandas.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ID           2484 non-null   int64
 1   Resume_str   2484 non-null   str  
 2   Resume_html  2484 non-null   str  
 3   Category     2484 non-null   str  
dtypes: int64(1), str(3)
memory usage: 52.3 MB


In [5]:
jobs.describe()

,country,country_code,date_added,has_expired,job_board,job_description,job_title,job_type,location,organization,page_url,salary,sector,uniq_id
count,22000,22000,122,22000,22000,22000,22000,20372,22000,15133,22000,3446,16806,22000
unique,1,1,78,1,1,18744,18759,39,8423,738,22000,1737,163,22000
top,United States of America,US,9/22/2016,No,jobs.monster.com,12N Horizontal Construction Engineers Job Desc...,Monster,Full Time,"Dallas, TX",Healthcare Services,http://jobview.monster.com/it-support-technici...,"40,000.00 - 100,000.00 $ /year",Experienced (Non-Manager),11d599f229a80023d2f40e7c52cd941e
freq,22000,22000,6,22000,22000,104,318,6757,646,1919,1,50,4594,1


In [6]:
jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 22000 entries, 0 to 21999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   country          22000 non-null  str  
 1   country_code     22000 non-null  str  
 2   date_added       122 non-null    str  
 3   has_expired      22000 non-null  str  
 4   job_board        22000 non-null  str  
 5   job_description  22000 non-null  str  
 6   job_title        22000 non-null  str  
 7   job_type         20372 non-null  str  
 8   location         22000 non-null  str  
 9   organization     15133 non-null  str  
 10  page_url         22000 non-null  str  
 11  salary           3446 non-null   str  
 12  sector           16806 non-null  str  
 13  uniq_id          22000 non-null  str  
dtypes: str(14)
memory usage: 67.1 MB


In [7]:
print("Job Shape :", jobs.shape)
print("Resume Shape :", resumes.shape)

Job Shape : (22000, 14)
Resume Shape : (2484, 4)


In [8]:
print(jobs.columns)

print(resumes.columns)

Index(['country', 'country_code', 'date_added', 'has_expired', 'job_board',
       'job_description', 'job_title', 'job_type', 'location', 'organization',
       'page_url', 'salary', 'sector', 'uniq_id'],
      dtype='str')
Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='str')


In [9]:
#jobs['job_title'] = jobs['job_description'].str.split().str[:3].str.join(" ")
# जर job_title 'monster' असेल तर description मधून role घेणे
jobs.loc[jobs['job_title'].str.lower() == 'monster', 'job_title'] = jobs.loc[jobs['job_title'].str.lower() == 'monster', 'job_description'].str.split().str[:3].str.join(" ")



In [10]:
#Keep Only Required Columns
# We only need the job_title and job_description columns 
# because the recommendation system compares the resume text with the job description. 
# Other columns like salary, country, and page URL are not required for similarity matching.

jobs = jobs[['job_title','job_description']]

In [11]:
resumes = resumes[['Resume_str','Category']]

In [12]:
jobs.head()

,job_title,job_description
0,IT Support Technician Job in Madison,TeamSoft is seeing an IT Support Specialist to...
1,Business Reporter/Editor Job in Madison,The Wisconsin State Journal is seeking a flexi...
2,Johnson & Johnson Family of Companies Job Appl...,Report this job About the Job DePuy Synthes Co...
3,Engineer - Quality Job in Dixon,Why Join Altec? If you’re considering a career...
4,Shift Supervisor - Part-Time Job in Camphill,Position ID# 76162 # Positions 1 State CT C...


In [13]:
resumes.head()

,Resume_str,Category
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,HR DIRECTOR Summary Over 2...,HR
3,HR SPECIALIST Summary Dedica...,HR
4,HR MANAGER Skill Highlights ...,HR


In [14]:
jobs.isnull().sum()

job_title          0
job_description    0
dtype: int64

In [15]:
resumes.isnull().sum()

Resume_str    0
Category      0
dtype: int64

In [16]:
print(jobs.shape)

print(resumes.shape)

(22000, 2)
(2484, 2)


In [17]:
#Text Cleaning


def clean_text(text):

    text = text.lower()

    text = re.sub(r'http\\S+',' ',text)#links remove karto

    text = re.sub(r'[^a-zA-Z ]',' ',text)#special symbol remove karto

    text = re.sub(r'\\s+',' ',text)#space remove karto

    return text

In [18]:
#Clean Job Description

jobs['job_description'] = jobs['job_description'].apply(clean_text)
jobs['job_title'] = jobs['job_title'].apply(clean_text)

In [19]:
resumes['Resume_str'] = resumes['Resume_str'].apply(clean_text)
resumes['Category'] = resumes['Category'].apply(clean_text)

In [20]:
jobs.head()

,job_title,job_description
0,it support technician job in madison,teamsoft is seeing an it support specialist to...
1,business reporter editor job in madison,the wisconsin state journal is seeking a flexi...
2,johnson johnson family of companies job appl...,report this job about the job depuy synthes co...
3,engineer quality job in dixon,why join altec if you re considering a career...
4,shift supervisor part time job in camphill,position id positions state ct c...


In [21]:
resumes.head()

,Resume_str,Category
0,hr administrator marketing associate ...,hr
1,hr specialist us hr operations ...,hr
2,hr director summary over ...,hr
3,hr specialist summary dedica...,hr
4,hr manager skill highlights ...,hr


In [22]:
jobs['job_text'] =jobs['job_text'] = (
    jobs['job_title'].str.lower() + " " +
    jobs['job_description'].str.lower()
)

In [23]:
jobs.head()

,job_title,job_description,job_text
0,it support technician job in madison,teamsoft is seeing an it support specialist to...,it support technician job in madison teamsoft ...
1,business reporter editor job in madison,the wisconsin state journal is seeking a flexi...,business reporter editor job in madison the wi...
2,johnson johnson family of companies job appl...,report this job about the job depuy synthes co...,johnson johnson family of companies job appl...
3,engineer quality job in dixon,why join altec if you re considering a career...,engineer quality job in dixon why join altec...
4,shift supervisor part time job in camphill,position id positions state ct c...,shift supervisor part time job in camphill p...


In [24]:
jobs1=jobs['job_text']

In [25]:
jobs.head()

,job_title,job_description,job_text
0,it support technician job in madison,teamsoft is seeing an it support specialist to...,it support technician job in madison teamsoft ...
1,business reporter editor job in madison,the wisconsin state journal is seeking a flexi...,business reporter editor job in madison the wi...
2,johnson johnson family of companies job appl...,report this job about the job depuy synthes co...,johnson johnson family of companies job appl...
3,engineer quality job in dixon,why join altec if you re considering a career...,engineer quality job in dixon why join altec...
4,shift supervisor part time job in camphill,position id positions state ct c...,shift supervisor part time job in camphill p...


In [26]:
#df=df["job_text"]

In [27]:
jobs['job_text'].str.len().describe()

count    22000.000000
mean      2620.575864
std       1722.286428
min         42.000000
25%       1434.750000
50%       2285.000000
75%       3394.000000
max      20249.000000
Name: job_text, dtype: float64

In [28]:
#TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [31]:
#Convert Job Text into Vectors

job_vectors = tfidf.fit_transform(jobs['job_text'])

In [32]:
job_vectors.shape

(22000, 90757)

In [33]:
#Resume ला TF-IDF मध्ये Convert करणे
resume_text = resumes['Resume_str'][0]


In [34]:
print(resume_text)

         hr administrator marketing associate  hr administrator       summary     dedicated customer service manager with     years of experience in hospitality and customer service management    respected builder and leader of customer focused teams  strives to instill a shared  enthusiastic commitment to customer service          highlights         focused on customer satisfaction  team management  marketing savvy  conflict resolution techniques     training and development  skilled multi tasker  client relations specialist           accomplishments      missouri dot supervisor training certification  certified by ihg in customer loyalty and marketing by segment   hilton worldwide general manager training certification  accomplished trainer for cross server hospitality systems such as    hilton onq      micros    opera pms     fidelio    opera    reservation system  ors      holidex    completed courses and seminars in customer service  sales strategies  inventory control  loss preve

In [35]:
#create resume vector
resume_vector = tfidf.transform([resume_text])

In [36]:
resume_vector.shape

(1, 90757)

In [37]:
#Cosine Similarity
#Resume ला सर्व 22000 jobs सोबत compare करेल.
similarity_scores = cosine_similarity(resume_vector, job_vectors)

In [38]:
similarity_scores.shape

(1, 22000)

In [39]:
#Top 5 Jobs Find
top_indices = similarity_scores[0].argsort()[-5:][::-1]


In [40]:
#Display Recommended Jobs
recommended_jobs= jobs.iloc[top_indices]

recommended_jobs[['job_title','job_description']]


,job_title,job_description
1461,events public relations assistant job in orl...,we have an immediate need for a public retail ...
16958,entry level assistant marketing advertising ...,about us the job window is seeking a entry lev...
8375,marketing manager entry level job in aurora,marketing manager entry levelour expanding co...
10891,public relations communications assistant ...,the job window has an immediate need for a pub...
7851,customer service client relations associate ...,do you have experience in the restaurant reta...


In [41]:
#Create Job Recommendation Function

def recommend_jobs(resume_text):

    # Step 1: Clean Resume Text
    resume_text = clean_text(resume_text)

    # Step 2: Convert Resume into TF-IDF vector
    resume_vector = tfidf.transform([resume_text])

    # Step 3: Calculate similarity with all jobs
    similarity_scores = cosine_similarity(
        resume_vector,
        job_vectors
    )

    # Step 4: Get top 5 matching jobs
    top_indices = similarity_scores[0].argsort()[-5:][::-1]

    # Step 5: Display jobs
    
    recommended_jobs = jobs.iloc[top_indices]

    return recommended_jobs[['job_title','job_description']]

    


In [42]:
import pdfplumber
def extract_resume_text(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + " "

    return text

In [43]:
import os

os.listdir("uploads")

['.ipynb_checkpoints', 'prachi resume.pdf (1).pdf']

In [44]:
pdf_path = "uploads/prachi resume.pdf (1).pdf"

In [45]:
resume_text = extract_resume_text(pdf_path)

print(resume_text)

Name: Prachi
CareerObjective:
Looking foropportunitiesin softwaredevelopmentanddata science.
Skills:
Python
SQL
MachineLearning
Natural LanguageProcessing
Pandas
NumPy
Data Analysis
HTML
CSS
Java
Projects:
Job Recommendation Systemusing NLPandMachineLearning.
Education:
Diploma ComputerEngineering
Experience:
Internship in Artificial IntelligenceandMachineLearning. 


In [46]:
recommended_jobs = recommend_jobs(resume_text)

recommended_jobs

,job_title,job_description
6815,title java,title java python developerlocation middle...
3660,java programmer with javascript and html job i...,experis is hiring a backend java programmer fo...
21492,marketing analyst job in cincinnati,email marketing analyst html css we are see...
13777,qa automation engineer selenium python java jo...,responsibilities kforce is working with a well...
18158,javascript html css ui developer job in irving,javascript html css ui developerdetailslocat...


In [47]:
recommend_jobs(resume_text)

,job_title,job_description
6815,title java,title java python developerlocation middle...
3660,java programmer with javascript and html job i...,experis is hiring a backend java programmer fo...
21492,marketing analyst job in cincinnati,email marketing analyst html css we are see...
13777,qa automation engineer selenium python java jo...,responsibilities kforce is working with a well...
18158,javascript html css ui developer job in irving,javascript html css ui developerdetailslocat...


In [48]:
#Improve Recommendation

skills = [
    'python',
    'sql',
    'machine learning',
    'nlp',
    'pandas',
    'numpy',
    'java',
    'html',
    'css'
]


In [49]:
def extract_skills(text):

    text = text.lower()

    found_skills = []

    for skill in skills:
        if skill in text:
            found_skills.append(skill)

    return found_skills

In [50]:
resume_skills = extract_skills(resume_text)

resume_skills

['python', 'sql', 'nlp', 'pandas', 'numpy', 'java', 'html', 'css']

In [51]:
import pickle
import gzip

# कमी jobs घ्या
small_jobs = jobs.head(1000)

small_vectors = job_vectors[:1000]


# Compressed save करा
with gzip.open("jobs.pkl.gz", "wb") as file:
    pickle.dump(small_jobs, file)


with gzip.open("job_vectors.pkl.gz", "wb") as file:
    pickle.dump(small_vectors, file)

In [52]:
# import pickle

# with open("tfidf.pkl","wb") as file:
#     pickle.dump(tfidf,file)


#jobs.pkl.gz → यात job data (उदा. job titles, descriptions इ.)

# job_vectors.pkl.gz → यात त्या jobs चे vectorized रूप (उदा. TF-IDF vectors, embeddings इ.)

In [53]:
# with open("job_vectors.pkl","wb") as file:
#     pickle.dump(job_vectors,file)

In [54]:
jobs.to_pickle("jobs.pkl")